# Camera-aware score normalization on MEVID-SC (CPU, no labels at test time)

**Idea:** a similarity of 0.6 does not mean the same thing on every camera pair, so one global threshold over-merges on some pairs and under-merges on others, and thresholds set on one group of people misfire on the other. We rescale each score relative to what is typical **for that camera pair**, with no labels, so it also works on new footage.

- **Camera-pair z-norm:** each score relative to the mean and spread of all scores between the same two cameras.
- **Camera S-norm:** relative to how each clip scores against the other camera, averaged from both sides (score normalization from speaker verification).

**Cell 4:** fewer IDs per person at the same precision? **Cell 5:** do thresholds now transfer (promised close to delivered), with the point-threshold baseline and ILRC calibrated on the other group, the train people, or both.

Needs the train fingerprints (already on Drive). CPU runtime, about 5 minutes. Results go to `MEVID-results/CameraNorm-SameClothes`.

In [ ]:
# 1. Load test + train fingerprints (CSCI, the OSNet baseline) and the groups
import os, glob, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0])

CS = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
           'train_templates_csci_sc.npz',
           f'{SD}/**/CSCI-SameClothes/train_templates_csci_sc.npz'])
TR = find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
           'train_templates_tracker_sc.npz',
           f'{SD}/**/Tracker-SameClothes/train_templates_tracker_sc.npz'])
OUT = os.path.join(os.path.dirname(CS), 'CameraNorm-SameClothes')
os.makedirs(OUT, exist_ok=True)


def unit(x):
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True),
                           1e-8)).astype(np.float32)


te = np.load(f'{CS}/test_templates_csci_sc.npz')
tr = np.load(f'{CS}/train_templates_csci_sc.npz')
tte = np.load(f'{TR}/test_templates_tracker_sc.npz')
ttr = np.load(f'{TR}/train_templates_tracker_sc.npz')
pid, cam = te['pid'], te['cam']
pid_tr, cam_tr = tr['pid'], tr['cam']
DATA = {'CSCI': (unit(tr['templates']), unit(te['templates'])),
        'OSNet baseline (OSNet)': (unit(ttr['zs']), unit(tte['zs']))}
proto = json.load(open(f'{CS}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])
SETTINGS = {'person_error_limit': 0.05, 'delta': 0.10,
            'alphas': [0.05, 0.10], 'bootstrap': 2000,
            'pair_precision_target': 0.95, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print('Test clips', len(pid), '| train clips', len(pid_tr),
      '| test cameras', len(np.unique(cam)), '| results ->', OUT)


In [ ]:
# 2. Camera-aware score normalization (label-free) + ID metrics
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment


def cam_pair_znorm(S, c):
    """each score relative to the mean and spread of ALL scores between
    the same two cameras (diagonal excluded)."""
    cams = np.unique(c); ci = np.searchsorted(cams, c)
    off = ~np.eye(len(S), dtype=bool)
    Z = np.zeros_like(S)
    gmu, gsd = S[off].mean(), S[off].std()
    for a in range(len(cams)):
        for b in range(a, len(cams)):
            m = np.outer(ci == a, ci == b) | np.outer(ci == b, ci == a)
            v = S[m & off]
            mu, sd = (v.mean(), v.std()) if len(v) > 10 else (gmu, gsd)
            Z[m] = (S[m] - mu) / max(sd, 1e-6)
    return Z


def cam_snorm(S, c):
    """S-norm by camera: score of (i, j) relative to how i scores against
    j's camera and how j scores against i's camera, averaged."""
    cams = np.unique(c); ci = np.searchsorted(cams, c)
    S2 = S.astype(np.float64).copy(); np.fill_diagonal(S2, np.nan)
    mu = np.zeros((len(S), len(cams))); sd = np.ones_like(mu)
    gmu, gsd = np.nanmean(S2), np.nanstd(S2)
    for k in range(len(cams)):
        blk = S2[:, ci == k]
        cnt = np.sum(~np.isnan(blk), 1)
        m_ = np.nanmean(np.where(cnt[:, None] > 0, blk, gmu), 1)
        s_ = np.nanstd(np.where(cnt[:, None] > 0, blk, gmu), 1)
        small = cnt < 5
        m_[small], s_[small] = gmu, gsd
        mu[:, k], sd[:, k] = m_, np.maximum(s_, 1e-6)
    Zi = (S - mu[:, ci]) / sd[:, ci]
    return 0.5 * (Zi + Zi.T)


def to_dist(Z):
    """fixed (group-independent) map from normalized score to distance."""
    D = 1 / (1 + np.exp(np.clip(Z, -50, 50)))
    np.fill_diagonal(D, 0)
    return D


SCORES = {'similarity': lambda S, c: 1 - S,
          'camera-pair z-norm': lambda S, c: to_dist(cam_pair_znorm(S, c)),
          'camera S-norm': lambda S, c: to_dist(cam_snorm(S, c))}
GRIDS = {'similarity': np.round(np.arange(0.01, 0.951, 0.01), 2),
         'camera-pair z-norm': np.geomspace(1e-6, 0.5, 120),
         'camera S-norm': np.geomspace(1e-6, 0.5, 120)}


def _cont(pred, p):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1))
    np.add.at(M, (pi, ci), 1)
    return M, pi, ci


def person_losses(pred, p):
    M, _, _ = _cont(pred, p)
    nc = M.sum(0)
    tot = (M * (nc - 1)).sum(1)
    wrong = (M * (nc - M)).sum(1)
    return np.where(tot > 0, wrong / np.maximum(tot, 1), 0.0)


def score(pred, p, c):
    M, pi, ci = _cont(pred, p)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0)
    tp = (M * (M - 1)).sum()
    same_id = (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    K = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(K, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (K.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (K ** 2).sum((1, 2))).sum()
    lp = person_losses(pred, p)
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': (100 * tp / same_id if same_id
                                 else np.nan),
            'cross-camera recall %': 100 * x_same / max(1, x_tot),
            'mean wrong-link share %': 100 * lp.mean()}


def sweep(D, p, c, grid, with_scores=True):
    D = np.clip((D + D.T) / 2, 0, None)
    Z = linkage(D[np.triu_indices(len(D), 1)], 'average')
    losses, scores = {}, {}
    for d in grid:
        lab = fcluster(Z, d, 'distance')
        losses[d] = person_losses(lab, p)
        if with_scores:
            scores[d] = score(lab, p, c)
    return losses, scores


In [ ]:
# 3. Global IDs at every threshold for each backbone x score, on both
#    test groups and on the train people (two halves, as a calibration
#    pool from OTHER cameras). Normalization uses only each set's own
#    clips and cameras, no labels.
rng = np.random.default_rng(SETTINGS['seed'])
tr_people = np.unique(pid_tr)
half = set(rng.choice(tr_people, len(tr_people) // 2, replace=False))
h = np.array([q in half for q in pid_tr])
RUN, rows = {}, []
for bb, (Xtr, Xte) in DATA.items():
    t0 = time.time()
    R = {'test': {}, 'train': {}}
    for g in 'AB':
        ig = np.flatnonzero(fold == g)
        S = Xte[ig] @ Xte[ig].T
        R['test'][g] = {s: sweep(f(S, cam[ig]), pid[ig], cam[ig], GRIDS[s])
                        for s, f in SCORES.items()}
    for s, f in SCORES.items():
        parts = []
        for part in [h, ~h]:
            ip = np.flatnonzero(part)
            S = Xtr[ip] @ Xtr[ip].T
            parts.append(sweep(f(S, cam_tr[ip]), pid_tr[ip], cam_tr[ip],
                               GRIDS[s], with_scores=False)[0])
        R['train'][s] = {d: np.concatenate([p_[d] for p_ in parts])
                         for d in GRIDS[s]}
    RUN[bb] = R
    for g, bys in R['test'].items():
        for s, (_, sc) in bys.items():
            for d, v in sc.items():
                rows.append({'backbone': bb, 'group': g, 'score': s,
                             'distance': d, **v})
    print(f'{bb}: done in {time.time() - t0:.0f} s')
SW = pd.DataFrame(rows)
SW.to_csv(f'{OUT}/sweep_test_groups.csv', index=False)
print('Saved to', OUT)


In [ ]:
# 4. FAIR COMPARISON at matched precision (diagnostic): does camera
#    normalization give fewer IDs per person at the same precision?
import matplotlib.pyplot as plt
rows = []
for (bb, g, s), d in SW.groupby(['backbone', 'group', 'score'],
                                sort=False):
    for lvl in [90, 95]:
        ok = d[d['pair precision %'] >= lvl]
        if len(ok):
            b = ok.loc[ok['IDF1 %'].idxmax()]
            rows.append({'backbone': bb, 'precision level %': lvl,
                         'score': s, 'IDF1 %': b['IDF1 %'],
                         'IDs per person': b['IDs per person'],
                         'cross-camera recall %':
                             b['cross-camera recall %']})
fair = (pd.DataFrame(rows).groupby(['backbone', 'precision level %',
                                    'score'], sort=False)
        [['IDF1 %', 'IDs per person', 'cross-camera recall %']]
        .mean().round(1))
fair.to_csv(f'{OUT}/fair_comparison.csv')
print('AT MATCHED PRECISION (mean of groups A and B)')
display(fair)

fig, axes = plt.subplots(1, len(RUN), figsize=(12, 4.5))
style = {'similarity': '--', 'camera-pair z-norm': '-',
         'camera S-norm': ':'}
for ax, bb in zip(np.atleast_1d(axes), RUN):
    for (s, g), d in SW[SW.backbone == bb].groupby(['score', 'group']):
        d = d.dropna(subset=['pair precision %'])
        ax.plot(d['pair precision %'], d['IDs per person'], style[s],
                label=f'{s} (group {g})')
    ax.set_title(bb); ax.set_xlabel('delivered pair precision %')
    ax.set_ylabel('IDs per person (lower is better)')
    ax.set_xlim(70, 100.5); ax.set_ylim(0.9, 12); ax.grid(alpha=.3)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUT}/curve_camera_norm.png', dpi=150)
plt.show()


In [ ]:
# 5. DO THRESHOLDS NOW TRANSFER? Calibrate on one pool, test on the
#    held-out group. Rules: point threshold (pair precision >= 95% on the
#    other group) and ILRC (person-bootstrap 90% bound on the mean
#    wrong-link share <= alpha; pools: other group, train people, both).
B, rows = SETTINGS['bootstrap'], []
rng = np.random.default_rng(SETTINGS['seed'])


def ilrc(L, grid, alpha):
    n = len(L[grid[0]])
    idx = rng.integers(0, n, size=(B, n))
    best = bnd = None
    for d in grid:
        ucb = np.quantile(L[d][idx].mean(1), 1 - SETTINGS['delta'])
        if ucb > alpha:
            break
        best, bnd = d, ucb
    return best, bnd


for bb, R in RUN.items():
    for cal, ev in [('A', 'B'), ('B', 'A')]:
        ie = np.flatnonzero(fold == ev)
        none = score(np.arange(len(ie)), pid[ie], cam[ie])
        for s in SCORES:
            grid = GRIDS[s]
            cL, cS = R['test'][cal][s]
            _, eS = R['test'][ev][s]
            d = None
            for x in grid:
                pp = cS[x]['pair precision %']
                if pp == pp and pp >= 100 * SETTINGS['pair_precision_target']:
                    d = x
            e = eS[d] if d is not None else none
            rows.append({'backbone': bb, 'score': s,
                         'rule': 'point threshold', 'pool': 'group',
                         'alpha %': np.nan, 'direction': f'{cal}->{ev}',
                         'promised pair precision %':
                             cS[d]['pair precision %'] if d is not None
                             else np.nan, **e})
            pools = {'group': cL, 'train': R['train'][s],
                     'group + train': {x: np.concatenate(
                         [cL[x], R['train'][s][x]]) for x in grid}}
            for pn, L in pools.items():
                for a in SETTINGS['alphas']:
                    d, bnd = ilrc(L, grid, a)
                    e = eS[d] if d is not None else none
                    rows.append({'backbone': bb, 'score': s, 'rule': 'ILRC',
                                 'pool': pn, 'alpha %': 100 * a,
                                 'direction': f'{cal}->{ev}',
                                 'promised wrong-link %':
                                     np.nan if bnd is None else 100 * bnd,
                                 **e})
res = pd.DataFrame(rows)
res['held'] = np.where(res['alpha %'].isna(), np.nan,
                       (res['mean wrong-link share %']
                        <= res['alpha %'].fillna(0)).astype(float))
res.to_csv(f'{OUT}/calibration_transfer_results.csv', index=False)
cols = ['IDF1 %', 'IDs per person', 'pair precision %',
        'promised pair precision %', 'mean wrong-link share %',
        'promised wrong-link %', 'held']
summ = (res.groupby(['backbone', 'score', 'rule', 'pool', 'alpha %'],
                    dropna=False, sort=False)[cols].mean().round(1))
summ.to_csv(f'{OUT}/calibration_transfer_summary.csv')
print('HELD-OUT GROUP, mean of A->B and B->A')
display(summ)
